# Lean variance analysis for `merged.csv`

This notebook is intentionally small: load `merged.csv`, apply the requested preprocessing stub, run train/test-aware statistical tests, plot raw/FDR-corrected p-value distributions, and plot variance explained.


In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import scipy.stats as stats
import matplotlib.pyplot as plt

plt.rcParams["svg.fonttype"] = "none"

DATA_PATH = Path("~/Desktop/plm_paper/merged.csv").expanduser()
OUT_DIR = Path("~/Desktop/plm_paper/refined_figures/supp_figure_var_analysis").expanduser()

# Set this to a CSV/TSV/parquet file to read a precomputed variance-explained table.
# Expected columns: dataset, feature or var, factor, var_explained.
VAR_EXPLAINED_PATH = None
CALCULATE_VARIANCE_EXPLAINED_IF_MISSING = True

SAVE_TABLES = True
SAVE_FIGURES = True

FEATURE_COLUMNS = ["roc", "top_100_pct", "correlation"]
MIN_GROUPS = 3
MIN_POINTS_PER_GROUP = 2
MIN_POINTS_FOR_ANOVA = 5


## Load and preprocess

Keep this function as the single preprocessing stub for `merged.csv` before analysis.


In [ ]:
def preprocess_merged_csv(df):
    """Preprocess merged.csv before any analysis."""
    df = df.copy()

    # Requested filtering stub.
    df = df[~df["model_name"].isin(["one_hot", "linreg"])]
    df = df[df["clf_type"] == "mlp"]

    return df.reset_index(drop=True)


raw_df = pd.read_csv(DATA_PATH)
df = preprocess_merged_csv(raw_df)

print(f"raw rows: {len(raw_df):,}")
print(f"preprocessed rows: {len(df):,}")
display(df.head())


In [ ]:
DATASET_LABELS = {
    "gfp": "GFP",
    "pard3": "PARD3",
    "nmt": "NMT",
    "gcn": "GCN4",
    "gcn4": "GCN4",
    "lov": "LOV",
    "his": "HIS",
    "his2": "HIS2",
    "his5": "HIS5",
    "casp": "CASP",
}

FEATURE_LABELS = {
    "roc": "ROC-AUC",
    "top_100_pct": "Top 100%",
    "correlation": "Correlation",
}


def available_features(sub_df, feature_columns=FEATURE_COLUMNS):
    return [col for col in feature_columns if col in sub_df.columns and sub_df[col].notna().any()]


def display_dataset(value):
    return DATASET_LABELS.get(str(value), str(value).upper())


def display_feature(value):
    return FEATURE_LABELS.get(str(value), str(value).replace("_", " ").title())


def fdr_bh(pvalues):
    pvalues = np.asarray(pvalues, dtype=float)
    adjusted = np.full(pvalues.shape, np.nan, dtype=float)
    valid = np.isfinite(pvalues)
    if valid.sum() == 0:
        return adjusted

    valid_pvalues = pvalues[valid]
    order = np.argsort(valid_pvalues)
    ranked = valid_pvalues[order]
    n = len(ranked)

    ranked_adjusted = np.empty(n, dtype=float)
    running_min = 1.0
    for i in range(n - 1, -1, -1):
        rank = i + 1
        running_min = min(running_min, ranked[i] * n / rank, 1.0)
        ranked_adjusted[i] = running_min

    restored = np.empty(n, dtype=float)
    restored[order] = ranked_adjusted
    adjusted[valid] = restored
    return adjusted


def add_fdr_by(df, p_col="P_value", group_cols=("analysis",)):
    df = df.copy()
    df["P_value_fdr_bh"] = np.nan
    for _, idx in df.groupby(list(group_cols), dropna=False).groups.items():
        df.loc[idx, "P_value_fdr_bh"] = fdr_bh(df.loc[idx, p_col].to_numpy())
    df["FDR_corrected"] = df["P_value_fdr_bh"]
    return df


## Statistical tests

Train/test logic here means:

- `same_train_mutations_compare_models`: within each dataset and train-mutation count, compare `model_name` groups.
- `same_test_mutations_compare_models`: within each dataset and test-mutation count, compare `model_name` groups.
- `same_test_mutations_compare_train_mutations`: within each dataset and test-mutation count, compare `train_mutations` groups.

The plots below use ANOVA p-values. Kruskal-Wallis results are kept in the output tables for inspection.


In [ ]:
def run_group_tests(
    df,
    *,
    analysis,
    comparison_type,
    compare_col,
    fixed_cols,
    feature_columns=FEATURE_COLUMNS,
    min_groups_for_anova=MIN_GROUPS,
    min_points_per_group=MIN_POINTS_PER_GROUP,
    min_points_for_anova=MIN_POINTS_FOR_ANOVA,
    permutation_iterations=10000,
):
    rows = []

    for fixed_values, sub_df in df.groupby(fixed_cols, dropna=False):
        if not isinstance(fixed_values, tuple):
            fixed_values = (fixed_values,)
        fixed = dict(zip(fixed_cols, fixed_values))

        for feature in available_features(sub_df, feature_columns):
            groups = []
            group_names = []
            for group_name, group_df in sub_df.groupby(compare_col, dropna=False):
                values = group_df[feature].dropna().to_numpy(dtype=float)
                if len(values) >= min_points_per_group:
                    groups.append(values)
                    group_names.append(group_name)

            row = {
                "analysis": analysis,
                "comparison_type": comparison_type,
                "feature": feature,
                "compare_col": compare_col,
                "fixed_cols": ",".join(fixed_cols),
                "n_groups": len(groups),
                "n_points": int(sum(len(g) for g in groups)),
                **fixed,
            }

            f_stat = p_value = k_stat = k_p_value = np.nan
            test_type = None

            if len(groups) > 1 and all(len(group) > 1 for group in groups):
                with warnings.catch_warnings():
                    warnings.simplefilter("ignore", stats.ConstantInputWarning)
                    f_stat, p_value = stats.f_oneway(*groups)
                try:
                    k_stat, k_p_value = stats.kruskal(*groups)
                except ValueError:
                    k_stat, k_p_value = np.nan, np.nan

                test_type = "ANOVA"
                if len(groups[0]) < min_points_for_anova or len(groups) < min_groups_for_anova:
                    test_type = "permutation"
                    f_obs = f_stat
                    f_perm = []
                    group_array = np.array(groups)
                    for _ in range(permutation_iterations):
                        permuted_groups = np.random.permutation(group_array)
                        f_stat_perm, _ = stats.f_oneway(*permuted_groups)
                        f_perm.append(f_stat_perm)
                    p_value = 1 - np.sum(np.sort(np.array([f_perm]) <= f_obs)) / len(f_perm)
                    k_p_value = np.nan

            row.update({
                "test_type": test_type,
                "F_value": f_stat,
                "P_value": p_value,
                "Kruskal_H": k_stat,
                "Kruskal_P": k_p_value,
                "groups": ",".join(map(str, group_names)),
            })
            rows.append(row)

    return pd.DataFrame(rows)


In [ ]:
model_pvalue_df = pd.concat(
    [
        run_group_tests(
            df,
            analysis="same_train_mutations_compare_models",
            comparison_type="train",
            compare_col="model_name",
            fixed_cols=["dataset", "train_mutations"],
        ),
        run_group_tests(
            df,
            analysis="same_test_mutations_compare_models",
            comparison_type="test",
            compare_col="model_name",
            fixed_cols=["dataset", "test_mutations"],
        ),
    ],
    ignore_index=True,
)
model_pvalue_df = add_fdr_by(model_pvalue_df, group_cols=("comparison_type",))

train_mutation_pvalue_df = run_group_tests(
    df,
    analysis="same_test_mutations_compare_train_mutations",
    comparison_type="test",
    compare_col="train_mutations",
    fixed_cols=["dataset", "test_mutations"],
)
train_mutation_pvalue_df = add_fdr_by(train_mutation_pvalue_df, group_cols=("analysis",))

if SAVE_TABLES:
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    model_pvalue_df.to_csv(OUT_DIR / "merged_model_comparison_pvalues.csv", index=False)
    train_mutation_pvalue_df.to_csv(OUT_DIR / "merged_train_mutation_comparison_pvalues.csv", index=False)

print("model-comparison tests:", model_pvalue_df["P_value"].notna().sum(), "valid p-values")
print("train-mutation tests:", train_mutation_pvalue_df["P_value"].notna().sum(), "valid p-values")
display(model_pvalue_df.head())
display(train_mutation_pvalue_df.head())


## P-value distributions

In [ ]:
def make_pvalue_plot_df(pvalue_df, *, p_col="FDR_corrected", comparison_types=("all", "test", "train")):
    rows = []
    work_df = pvalue_df.copy()
    work_df["dataset_label"] = work_df["dataset"].map(display_dataset)

    combo = work_df[["feature", "dataset", "dataset_label"]].drop_duplicates()
    plot_order = combo.apply(lambda r: f"{r['feature']}\n{r['dataset_label']}", axis=1).tolist()
    plot_labels = combo.apply(lambda r: f"{r['dataset_label']}\n({display_feature(r['feature'])})", axis=1).tolist()

    for fd, pretty_label in zip(plot_order, plot_labels):
        feature, dataset_label = fd.split("\n", 1)
        base_mask = (work_df["feature"] == feature) & (work_df["dataset_label"] == dataset_label)
        for comp_type in comparison_types:
            if comp_type == "all":
                mask = base_mask
            else:
                mask = base_mask & (work_df["comparison_type"] == comp_type)
            for _, row in work_df.loc[mask].iterrows():
                test_type = row.get("test_type")
                rows.append({
                    "feature_dataset": fd,
                    "feature_dataset_y": pretty_label,
                    "comparison_type": comp_type,
                    "P_value": row[p_col],
                    "test_type": test_type.lower() if isinstance(test_type, str) else None,
                })

    plot_df = pd.DataFrame(rows)
    if len(plot_df):
        plot_df = plot_df[plot_df["comparison_type"].isin(comparison_types)]
        plot_df = plot_df[plot_df["feature_dataset"].isin(plot_order)]
    return plot_df, plot_order, plot_labels


def plot_pvalue_summary(
    pvalue_df,
    *,
    p_col="FDR_corrected",
    comparison_types=("all", "test", "train"),
    output_name=None,
    figsize=(8, 4),
):
    plot_df, plot_order, plot_labels = make_pvalue_plot_df(
        pvalue_df,
        p_col=p_col,
        comparison_types=comparison_types,
    )
    if plot_df.empty:
        print("No p-values to plot.")
        return None

    summary = (
        plot_df.groupby(["feature_dataset", "comparison_type"])["P_value"]
        .agg(["mean", "std"])
        .reset_index()
    )

    color_values = plt.get_cmap("Set1")(np.linspace(0, 1, max(3, len(comparison_types))))
    color_map = dict(zip(comparison_types, color_values))
    width = 0.25
    if len(comparison_types) == 1:
        dodge_pos = {comparison_types[0]: 0.0}
    else:
        dodge_pos = dict(zip(comparison_types, np.linspace(-width, width, len(comparison_types))))

    rng = np.random.default_rng(1)
    fig, ax = plt.subplots(figsize=figsize)

    for i, fd in enumerate(plot_order):
        for comp_type in comparison_types:
            sub = summary[(summary["feature_dataset"] == fd) & (summary["comparison_type"] == comp_type)]
            if sub.empty:
                continue
            mean = sub["mean"].iloc[0]
            std = sub["std"].iloc[0]
            ax.errorbar(
                i + dodge_pos[comp_type],
                mean,
                yerr=0.0 if pd.isna(std) else std,
                fmt="o",
                markersize=8,
                color=color_map[comp_type],
                elinewidth=2,
                capsize=1,
                alpha=0.7,
                markeredgewidth=0,
                zorder=4,
            )

    marker_dict = {"anova": "o", "permutation": "s", None: "o"}
    label_dict = {"anova": "ANOVA", "permutation": "Permutation"}
    seen_test_types = set()

    for i, fd in enumerate(plot_order):
        for comp_type in comparison_types:
            pts = plot_df[(plot_df["feature_dataset"] == fd) & (plot_df["comparison_type"] == comp_type)]
            for test_type, pts_tt in pts.groupby("test_type", dropna=False):
                marker = marker_dict.get(test_type, "o")
                x = i + dodge_pos[comp_type] + rng.uniform(-0.09, 0.09, len(pts_tt))
                ax.scatter(
                    x,
                    pts_tt["P_value"].to_numpy(),
                    s=45,
                    marker=marker,
                    color="black",
                    alpha=0.85,
                    linewidth=0.7,
                    edgecolors="black",
                    zorder=3,
                )
                seen_test_types.add(test_type)

    from matplotlib.lines import Line2D

    comp_handles = [
        Line2D([0], [0], marker="o", color="w", markerfacecolor=color_map[c], markersize=8, linestyle="None", label=c)
        for c in comparison_types
    ]
    test_handles = [
        Line2D([0], [0], marker=marker_dict[t], color="black", markerfacecolor="black", markersize=8, linestyle="None", label=label_dict[t])
        for t in ("anova", "permutation")
        if t in seen_test_types
    ]

    ax.grid(True, which="major", linestyle="--", linewidth=0.25, alpha=0.7)
    ax.spines["right"].set_visible(False)
    ax.spines["top"].set_visible(False)
    ax.set_xticks(range(len(plot_order)))
    ax.set_xticklabels(plot_labels, rotation=30, ha="right")
    ax.set_ylabel("P-value (Corrected)" if p_col == "FDR_corrected" else "P-value")
    ax.set_xlabel("")
    ax.axhline(0.05, color="black", linestyle="--", lw=1)
    ax.legend(handles=comp_handles + test_handles, title="Comparison / Test Type", loc="best")
    fig.tight_layout()

    if SAVE_FIGURES and output_name is not None:
        OUT_DIR.mkdir(parents=True, exist_ok=True)
        fig.savefig(OUT_DIR / output_name, format="svg", bbox_inches="tight")

    return fig


In [ ]:
plot_pvalue_summary(
    model_pvalue_df,
    p_col="FDR_corrected",
    comparison_types=("all", "test", "train"),
    output_name="pvalue_dist_corrected_fdr_new.svg",
)
plt.show()


In [ ]:
plot_pvalue_summary(
    train_mutation_pvalue_df,
    p_col="FDR_corrected",
    comparison_types=("test",),
    output_name="pvalue_dist_corrected_fdr_new_train_as_factor.svg",
)
plt.show()


## Cumulative p-value distributions

In [ ]:
def plot_pvalue_cdf(pvalue_df, p_col, *, comparison_types=("all", "test", "train"), output_name=None):
    fig, axes = plt.subplots(1, len(comparison_types), figsize=(2.8 * len(comparison_types), 2.7), sharey=True)
    if len(comparison_types) == 1:
        axes = [axes]

    for ax, comp_type in zip(axes, comparison_types):
        if comp_type == "all":
            values = pvalue_df[p_col]
        else:
            values = pvalue_df.loc[pvalue_df["comparison_type"] == comp_type, p_col]

        pvalues = values.dropna().sort_values().to_numpy()
        if len(pvalues):
            cumulative = 1 - np.arange(1, len(pvalues) + 1) / len(pvalues)
            significant = 100 * np.mean(pvalues < 0.05)
            ax.step(pvalues[::-1], cumulative[::-1], where="post", lw=2)
            ax.set_title(f"{comp_type}\n{significant:.1f}% < 0.05")
        else:
            ax.set_title(f"{comp_type}\nno p-values")

        ax.axvline(0.05, color="black", linestyle="--", lw=1)
        ax.set_xlim(1.05, -0.02)
        ax.set_ylim(-0.02, 1.02)
        ax.set_xlabel(p_col)
        ax.grid(True, linestyle="--", linewidth=0.25, alpha=0.7)
        ax.spines["right"].set_visible(False)
        ax.spines["top"].set_visible(False)

    axes[0].set_ylabel("Fraction >= p-value")
    fig.tight_layout()

    if SAVE_FIGURES and output_name is not None:
        OUT_DIR.mkdir(parents=True, exist_ok=True)
        fig.savefig(OUT_DIR / output_name, format="svg", bbox_inches="tight")

    return fig


plot_pvalue_cdf(
    model_pvalue_df,
    "P_value",
    comparison_types=("all", "test", "train"),
    output_name="pvalue_fractions_new.svg",
)
plot_pvalue_cdf(
    model_pvalue_df,
    "FDR_corrected",
    comparison_types=("all", "test", "train"),
    output_name="pvalue_fractions_corrected_fdr_new.svg",
)
plot_pvalue_cdf(
    train_mutation_pvalue_df,
    "P_value",
    comparison_types=("test",),
    output_name="by_train_mutations_pvalue_fractions_new.svg",
)
plot_pvalue_cdf(
    train_mutation_pvalue_df,
    "FDR_corrected",
    comparison_types=("test",),
    output_name="by_train_mutations_pvalue_fractions_corrected_fdr_new.svg",
)
plt.show()


## F-value distribution

In [ ]:
def plot_fvalue_distribution(pvalue_df, *, comparison_types=("all", "test", "train"), output_name=None):
    fig, axes = plt.subplots(1, len(comparison_types), figsize=(2.8 * len(comparison_types), 2.7), sharey=True)
    if len(comparison_types) == 1:
        axes = [axes]

    for ax, comp_type in zip(axes, comparison_types):
        if comp_type == "all":
            values = pvalue_df["F_value"]
        else:
            values = pvalue_df.loc[pvalue_df["comparison_type"] == comp_type, "F_value"]

        log_values = np.log10(values.replace([np.inf, -np.inf], np.nan).dropna())
        log_values = log_values[np.isfinite(log_values)]
        if len(log_values):
            ax.hist(log_values, bins=15, zorder=3)
        ax.axvline(0, color="black", linestyle="--", lw=1, zorder=4)
        ax.set_title(comp_type)
        ax.set_xlabel("log10(F-value)")
        ax.grid(True, which="major", linestyle="--", linewidth=0.25, alpha=0.7, zorder=2)
        ax.spines["right"].set_visible(False)
        ax.spines["top"].set_visible(False)

    axes[0].set_ylabel("Count")
    fig.tight_layout()

    if SAVE_FIGURES and output_name is not None:
        OUT_DIR.mkdir(parents=True, exist_ok=True)
        fig.savefig(OUT_DIR / output_name, format="svg", bbox_inches="tight")

    return fig


plot_fvalue_distribution(
    model_pvalue_df,
    comparison_types=("all", "test", "train"),
    output_name="f_value_distribution_new.svg",
)
plot_fvalue_distribution(
    train_mutation_pvalue_df,
    comparison_types=("test",),
    output_name="f_value_distribution_train_as_factor.svg",
)
plt.show()


## Variance explained

By default this calculates a lean one-way eta-squared value (`SS_factor / SS_total`) independently inside each dataset/test-mutation regime. `test_mutations` is recorded as the regime, not used as a factor. To read an existing table instead, set `VAR_EXPLAINED_PATH` in the config cell.


In [ ]:
def read_table(path):
    path = Path(path).expanduser()
    if path.suffix == ".parquet":
        return pd.read_parquet(path)
    if path.suffix in {".tsv", ".tab"}:
        return pd.read_csv(path, sep="	")
    return pd.read_csv(path)


def one_way_variance_explained(sub_df, feature, factor):
    values = sub_df[[feature, factor]].dropna()
    if values[feature].nunique() <= 1 or values[factor].nunique() <= 1:
        return np.nan

    y = values[feature].to_numpy(dtype=float)
    overall_mean = y.mean()
    total_ss = np.sum((y - overall_mean) ** 2)
    if total_ss == 0:
        return np.nan

    grouped = values.groupby(factor)[feature].agg(["mean", "count"])
    factor_ss = np.sum(grouped["count"] * (grouped["mean"] - overall_mean) ** 2)
    return factor_ss / total_ss


def calculate_variance_explained(df, factors=("model_name", "train_mutations", "budget", "scale")):
    rows = []
    factors = [factor for factor in factors if factor in df.columns]

    for (dataset, test_mutations), regime_df in df.groupby(["dataset", "test_mutations"]):
        for feature in available_features(regime_df):
            for factor in factors:
                rows.append({
                    "dataset": dataset,
                    "test_mutations": test_mutations,
                    "feature": feature,
                    "factor": factor,
                    "var_explained": one_way_variance_explained(regime_df, feature, factor),
                })
    return pd.DataFrame(rows)


def load_or_calculate_variance_explained():
    if VAR_EXPLAINED_PATH is not None:
        var_df = read_table(VAR_EXPLAINED_PATH)
    elif CALCULATE_VARIANCE_EXPLAINED_IF_MISSING:
        var_df = calculate_variance_explained(df)
    else:
        return pd.DataFrame()

    var_df = var_df.rename(columns={"var": "feature"}).copy()
    required = {"dataset", "feature", "factor", "var_explained"}
    missing = required - set(var_df.columns)
    if missing:
        raise ValueError(f"variance-explained table is missing columns: {sorted(missing)}")
    return var_df


var_explained_df = load_or_calculate_variance_explained()

if SAVE_TABLES and len(var_explained_df):
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    var_explained_df.to_csv(OUT_DIR / "merged_variance_explained.csv", index=False)

display(var_explained_df)


In [ ]:
def plot_variance_explained(var_df, output_name="variance_explained_by_factor_barplot.svg"):
    if var_df.empty:
        print("No variance-explained data to plot.")
        return None

    df_plot = var_df.rename(columns={"var": "feature"}).copy()
    df_plot = df_plot.dropna(subset=["var_explained"])
    if df_plot.empty:
        print("Variance-explained data only contains NaN values.")
        return None

    if df_plot["var_explained"].max() <= 1.5:
        df_plot["var_explained"] = 100 * df_plot["var_explained"]

    df_plot["dataset_label"] = df_plot["dataset"].map(display_dataset)
    combo = df_plot[["feature", "dataset", "dataset_label"]].drop_duplicates().reset_index(drop=True)
    combo["feature_dataset"] = combo.apply(lambda r: f"{r['feature']}\n{r['dataset_label']}", axis=1)
    df_plot = df_plot.merge(combo, on=["feature", "dataset", "dataset_label"], how="left")

    plot_order = combo["feature_dataset"].tolist()
    plot_labels = combo.apply(lambda r: f"{r['dataset_label']}\n({display_feature(r['feature'])})", axis=1).tolist()

    factor_order = ["model_name", "budget", "train_mutations", "scale"]
    present_factors = [f for f in factor_order if f in set(df_plot["factor"])]
    present_factors += sorted(set(df_plot["factor"]) - set(present_factors))
    colors = plt.get_cmap("Set1")(np.linspace(0, 1, max(3, len(present_factors))))
    factor_color_map = dict(zip(present_factors, colors))

    factor_orders = {}
    for fd in plot_order:
        tmp = df_plot[df_plot["feature_dataset"] == fd]
        medians = tmp.groupby("factor")["var_explained"].median().to_dict()
        factor_orders[fd] = sorted(medians, key=lambda f: (medians[f], f))

    fig, ax = plt.subplots(figsize=(8.2, 3))
    rng = np.random.default_rng(1)
    bar_width = 0.8
    x_ticks = np.arange(len(plot_order))

    for x_pos, fd in enumerate(plot_order):
        vdf = df_plot[df_plot["feature_dataset"] == fd]
        factors = factor_orders[fd]
        offsets = np.linspace(-bar_width / 2, bar_width / 2, len(factors)) if len(factors) else []

        for factor, offset in zip(factors, offsets):
            vals = vdf.loc[vdf["factor"] == factor, "var_explained"].dropna().to_numpy()
            if len(vals) == 0:
                continue

            mean = vals.mean()
            std = vals.std()
            bar_x = x_pos + offset
            bar_unit_width = bar_width / len(factors)

            ax.bar(
                bar_x,
                mean,
                width=bar_unit_width * 0.95,
                color=factor_color_map.get(factor, "gray"),
                alpha=0.85,
                zorder=1,
                edgecolor="white",
                linewidth=0.8,
            )
            if std > 0:
                ax.vlines(bar_x, mean, mean + std, colors="black", lw=1.2, zorder=3)
                ax.hlines(mean + std, bar_x - 0.045, bar_x + 0.045, colors="black", lw=0.7, zorder=3)

            jitter_x = rng.normal(loc=bar_x, scale=bar_unit_width * 0.10, size=len(vals))
            ax.scatter(jitter_x, vals, color="black", alpha=0.35, s=16, zorder=4)

    from matplotlib.lines import Line2D

    handles = [
        Line2D([0], [0], color=factor_color_map[f], marker="o", linestyle="", markersize=8, label=f)
        for f in present_factors
        if any(f in factor_orders[fd] for fd in plot_order)
    ]
    ax.legend(handles=handles, frameon=False)
    ax.set_xticks(x_ticks)
    ax.set_xticklabels(plot_labels, rotation=30, ha="right", fontsize=9)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.set_ylabel("Variance Explained (%)")
    ax.set_xlabel("")
    ax.grid(True, axis="y", linestyle="--", linewidth=0.3, alpha=0.7)
    fig.tight_layout()

    if SAVE_FIGURES:
        OUT_DIR.mkdir(parents=True, exist_ok=True)
        fig.savefig(OUT_DIR / output_name, format="svg", bbox_inches="tight")

    return fig


plot_variance_explained(var_explained_df)
plt.show()
